In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import torch

import pickle

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample

import pandas as pd

device = torch.device("cuda")

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample

from pyro.infer.autoguide.initialization import init_to_median

import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample

from torchvision.datasets import ImageFolder

from dotenv import load_dotenv
import requests
import os

import warnings

class SmartPool(nn.Module):
    """
    A “smart” max‐pool that detects outliers (values > threshold) and, if desired,
    replaces them with the 2nd‐largest value in the window.
    """
    def __init__(
        self,
        kernel_size: int = 2,
        stride: int = 2,
        threshold: float = 10.0,
        detect_only: bool = False
    ):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.threshold = threshold
        self.detect_only = detect_only

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, C, H, W)
        N, C, H, W = x.shape
        ks, st = self.kernel_size, self.stride

        # Unfold into patches of shape (N, C, ks*ks, L) where L = #windows per image
        patches = F.unfold(x, kernel_size=ks, stride=st)  # → (N, C*ks*ks, L)
        patches = patches.view(N, C, ks*ks, -1)           # → (N, C, ks*ks, L)

        # Find top‐2 values in each patch
        top2_vals, _ = torch.topk(patches, 2, dim=2)      # → (N, C, 2, L)
        max1 = top2_vals[:, :, 0, :]                      # → (N, C, L)
        max2 = top2_vals[:, :, 1, :]                      # → (N, C, L)

        # Detect any spikes above threshold
        spikes = max1 > self.threshold                    # → (N, C, L)
        #if spikes.any():
        #    warnings.warn(f"SmartPool: detected {int(spikes.sum())} pooled values above threshold={self.threshold}")

        # If correction is off, just return the regular max‐pooled result
        if self.detect_only:
            out = max1
        else:
            # Replace each spike with the 2nd‐largest value
            out = torch.where(spikes, max2, max1)

        # Fold back to (N, C, H_out, W_out)
        H_out, W_out = (H // ks, W // ks)
        out = out.view(N, C * 1, -1)                      # → (N, C, L)
        out = out.view(N, C, H_out, W_out)
        return out

class BayesShipsCNNSmartpool(PyroModule):
    def __init__(
        self,
        num_classes=2,   # now 2 for Categorical
        device=torch.device("cuda"),
        activation='relu',
        prior_dist='gaussian',
        mu=0.0,
        b=1.0,
        prior_params=None,
        smartpool_switch = False,
        pool_threshold=10.0,
        pool_detect_only=False,
        dropout_switch=False,
        dropout_p=0.5
    ):
        super().__init__()
        self.device = device

        # Activation setup
        if isinstance(activation, str):
            act_map = {
                'relu': F.relu,
                'tanh': torch.tanh,
                'sigmoid': torch.sigmoid,
                'sinusoidal': torch.sin,
                'relu6': F.relu6,
                'leaky_relu': F.leaky_relu,
                'selu': F.selu,
                'wg': self.actWG,
                'rwg': self.actRWG,
            }
            self.activation_fn = act_map[activation]
        elif callable(activation):
            self.activation_fn = activation
        else:
            raise ValueError("activation must be a string or callable")

        # Prior setup
        self.prior_dist = prior_dist
        params = {'mu': mu, 'b': b} if prior_params is None else prior_params
        self.prior_mu = torch.tensor(params['mu'], device=device, dtype=torch.float32)
        self.prior_b  = torch.tensor(params['b'], device=device, dtype=torch.float32)

        print(f"[INFO] Using prior: {self.prior_dist} (mu={self.prior_mu.item()}, b={self.prior_b.item()})")

        # Layers
        self.conv1 = PyroModule[nn.Conv2d](3, 32, kernel_size=3, padding=1)
        self.conv1.weight = PyroSample(self._make_prior([32, 3, 3, 3]))
        self.conv1.bias   = PyroSample(self._make_prior([32]))

        self.conv2 = PyroModule[nn.Conv2d](32, 64, kernel_size=3, padding=1)
        self.conv2.weight = PyroSample(self._make_prior([64, 32, 3, 3]))
        self.conv2.bias   = PyroSample(self._make_prior([64]))

        if not smartpool_switch:
            self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        elif smartpool_switch:

            self.pool = SmartPool(
                kernel_size=2,
                stride=2,
                threshold=pool_threshold,
                detect_only=pool_detect_only
            )

        self.dropout_switch = dropout_switch

        if self.dropout_switch:
            self.dropout = nn.Dropout(p=dropout_p)

        self.fc1 = PyroModule[nn.Linear](64 * 16 * 16, num_classes)
        self.fc1.weight = PyroSample(self._make_prior([num_classes, 64 * 16 * 16]))
        self.fc1.bias   = PyroSample(self._make_prior([num_classes]))

    def actWG(self, x, alpha=1.0):
        return x * torch.exp(-alpha * x ** 2)

    def actRWG(self, x, alpha=1.0):
        wg = x * torch.exp(-alpha * x ** 2)
        return torch.max(torch.zeros_like(wg), wg)

    def _make_prior(self, shape):
        if self.prior_dist == 'gaussian':
            base = dist.Normal(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'laplace':
            base = dist.Laplace(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'uniform':
            base = dist.Uniform(-self.prior_b, self.prior_b)
        else:
            raise ValueError(f"Unsupported prior: {self.prior_dist}")
        return base.expand(shape).to_event(len(shape))

    def forward(self, x, y=None):
        x = self.activation_fn(self.conv1(x))
        x = self.pool(x)
        x = self.activation_fn(self.conv2(x))
        x = self.pool(x)

        if self.dropout_switch:
            x = self.dropout(x)

        x = x.view(x.size(0), -1)
        logits = self.fc1(x)  # shape [batch, 2]

        if y is not None:
            with pyro.plate("data", x.size(0)):
                pyro.sample("obs", dist.Categorical(logits=logits), obs=y)
        return logits

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

In [3]:
def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    #dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)
    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )
    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader

In [4]:
import os
import torch
import pyro
from tqdm import tqdm
import numpy as np

def train_svi_with_stats(
    model,
    guide,
    svi,
    train_loader,
    device,
    num_epochs=10,
    save_epochs=None,
    save_dir='results_guidemultivariate',
    model_filename_pattern='model_{activation}_{prior}_epoch_{epoch}_{timestamp}.pth',
    guide_filename_pattern='guide_{activation}_{prior}_epoch_{epoch}_{timestamp}.pth',
    param_store_filename_pattern='param_store_{activation}_{prior}_epoch_{epoch}_{timestamp}.pkl',
    accuracies_filename_pattern='accuracy_results_{activation}_{prior}_{timestamp}.csv',
    losses_filename_pattern='losses_{activation}_{prior}_{timestamp}.csv',
    model_config_filename_pattern='config_{activation}_{prior}_{timestamp}.json'
):
    """
    Train the SVI model, track losses/accuracies, and
    save artifacts only when accuracy improves, naming files
    like `model_relu_gaussian_epoch_3.pth`.
    """
    
    # Pull names off the model if available, else fall back
    #act_name  = getattr(model, 'activation', getattr(model, 'activation_name', 'act'))
    act_name = model.activation_fn.__name__ if hasattr(model.activation_fn, '__name__') else str(model.activation_fn)
    prior_name = getattr(model, 'prior_dist', 'prior')
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    os.makedirs(save_dir, exist_ok=True)
    save_epochs = set(save_epochs or range(1, num_epochs+1))

    pyro.clear_param_store()
    model.to(device)

    epoch_losses, epoch_accuracies, accuracy_epochs = [], [], []
    loc_stats = {'epochs': [], 'means': [], 'stds': []}
    scale_stats   = {'epochs': [], 'means': [], 'stds': []}
    best_acc = 0.0

    for epoch in range(1, num_epochs+1):
        model.train()
        total_loss = 0.0
        batches = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device).long()
            total_loss += svi.step(images, labels)
            batches += 1

        avg_loss = total_loss / batches
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch} - ELBO Loss: {avg_loss:.4f}")

        if epoch == 1 or epoch % 10 == 0 or epoch == num_epochs:
            model.eval(); guide.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, labels in tqdm(train_loader, desc=f"Acc check epoch {epoch}"):
                    images, labels = images.to(device), labels.to(device)
                    trace = pyro.poutine.trace(guide).get_trace(images)
                    replayed = pyro.poutine.replay(model, trace=trace)
                    logits = replayed(images)
                    preds = torch.argmax(logits, dim=1)
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

            acc = correct/total
            epoch_accuracies.append(acc); accuracy_epochs.append(epoch)
            print(f"Epoch {epoch} - Train Acc: {acc*100:.2f}%")

            # record stats...
            w_means, w_stds, b_means, b_stds = [], [], [], []
            for name, param in pyro.get_param_store().items():
                if 'loc' or 'low' in name:
                    w_means.append(param.mean().item()); w_stds.append(param.std(unbiased=False).item())
                elif 'scale' or 'width' in name:
                    b_means.append(param.mean().item()); b_stds.append(param.std(unbiased=False).item())
            loc_stats['epochs'].append(epoch)
            loc_stats['means'].append(w_means)
            loc_stats['stds'].append(w_stds)
            scale_stats['epochs'].append(epoch)
            scale_stats['means'].append(b_means)
            scale_stats['stds'].append(b_stds)

            #for name, param in pyro.get_param_store().items():
            #    if 'loc' in name or 'scale' in name:
            #        print(f"{name}: {param.detach().cpu().numpy()}")

            # only save when accuracy improves
            if acc > best_acc:
                best_acc = acc
                fname_model = model_filename_pattern.format(activation=act_name, prior=prior_name, epoch="best", timestamp=timestamp)
                fname_guide = guide_filename_pattern.format(activation=act_name, prior=prior_name, epoch="best", timestamp=timestamp)
                fname_ps    = param_store_filename_pattern.format(activation=act_name, prior=prior_name, epoch="best", timestamp=timestamp)

                #torch.save(model.state_dict(), os.path.join(save_dir, fname_model))
                #torch.save(guide.state_dict(), os.path.join(save_dir, fname_guide))
                pyro.get_param_store().save(os.path.join(save_dir, fname_ps))
                print(f"  ↳ Saved: {fname_model}, {fname_guide}, {fname_ps}")

    # save losses per epoch in a csv file, with consistent file naming
    accuracies_df = pd.DataFrame({
        'epoch': accuracy_epochs,
        'accuracy': epoch_accuracies
    })
    #accuracies_df.to_csv(os.path.join(save_dir,accuracies_filename_pattern.format(activation=act_name, prior=prior_name, timestamp=timestamp)), index=False)

    loss_df = pd.DataFrame({
        'epoch': list(range(1, epoch + 1)),
        'loss': epoch_losses
    })
    #loss_df.to_csv(os.path.join(save_dir,losses_filename_pattern.format(activation=act_name, prior=prior_name, timestamp=timestamp)), index=False)
            
    # save model configuration in a json file
    config = {
        'activation': act_name,
        'prior': prior_name,
        'num_epochs': num_epochs,
        'best_accuracy_at_epoch': accuracy_epochs[np.argmax(epoch_accuracies)],
        'best_accuracy': best_acc,
        'batch_size': train_loader.batch_size,
        'train_size': len(train_loader.dataset),
        'prior_params': {
            'mu': model.prior_mu.item(),
            'b': model.prior_b.item()
        },
    }
    config_filename = model_config_filename_pattern.format(activation=act_name, prior=prior_name, timestamp=timestamp)

    #with open(os.path.join(save_dir, config_filename), 'w') as f:
    #    import json
    #    json.dump(config, f, indent=4)
    #    print(f"Configuration saved to {config_filename}")

    return epoch_losses, epoch_accuracies, accuracy_epochs, loc_stats, scale_stats, os.path.join(save_dir, fname_model), os.path.join(save_dir, fname_guide), os.path.join(save_dir, fname_ps), timestamp

In [5]:
def plot_training_results_with_stats(losses, accuracies, accuracy_epochs, loc_stats, scale_stats, act_name, prior_name, timestamp):
    """Plot training results with weight and bias statistics"""
    plt.figure(figsize=(16, 12))
    
    # Plot 1: Training Loss
    plt.subplot(2, 2, 1)
    plt.plot(range(1, len(losses) + 1), losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('ELBO Loss')
    plt.grid(True)
    
    # Plot 2: Training Accuracy
    plt.subplot(2, 2, 2)
    plt.plot(accuracy_epochs, accuracies, 'o-')
    plt.title('Training Accuracy (Every 10 Epochs)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    
    # Plot 3: Weight Statistics Boxplot
    plt.subplot(2, 2, 3)
    loc_data = []
    loc_labels = []
    
    for i, epoch in enumerate(loc_stats['epochs']):
        # Combine means and stds for this epoch
        epoch_data = loc_stats['means'][i] + loc_stats['stds'][i]
        loc_data.append(epoch_data)
        loc_labels.append(f'Epoch {epoch}')
    
    if loc_data:
        bp1 = plt.boxplot(loc_data, labels=loc_labels, patch_artist=True)
        for patch in bp1['boxes']:
            patch.set_facecolor('lightblue')
    
    plt.title('LOC Statistics Distribution')
    plt.xlabel('Epoch')
    plt.ylabel('LOC Values')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # Plot 4: Scale Statistics Boxplot
    plt.subplot(2, 2, 4)
    scale_data = []
    scale_labels = []
    
    for i, epoch in enumerate(scale_stats['epochs']):
        # Combine means and stds for this epoch
        epoch_data = scale_stats['means'][i] + scale_stats['stds'][i]
        scale_data.append(epoch_data)
        scale_labels.append(f'Epoch {epoch}')
    
    if scale_data:
        #bp2 = plt.boxplot(scale_data, tick_labels=scale_labels, patch_artist=True)
        bp2 = plt.boxplot(scale_data, labels=scale_labels, patch_artist=True)
        for patch in bp2['boxes']:
            patch.set_facecolor('lightcoral')
    
    plt.title('SCALE Statistics Distribution')
    plt.xlabel('Epoch')
    plt.ylabel('SCALE Values')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    #plt.savefig(os.path.join(args.save_dir,f'bayesian_cnn_training_results_{act_name}_{prior_name}_{timestamp}.png'))
    plt.show()

In [6]:
import numpy as np
from sklearn.metrics import confusion_matrix


def predict_data(model, loader_of_interest, num_samples=10):
    model.eval()
    guide.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in tqdm(loader_of_interest, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)

            logits_mc = torch.zeros(num_samples, images.size(0), model.fc1.out_features).to(device)

            for i in range(num_samples):
                guide_trace = pyro.poutine.trace(guide).get_trace(images)
                replayed_model = pyro.poutine.replay(model, trace=guide_trace)
                logits = replayed_model(images)
                logits_mc[i] = logits

            avg_logits = logits_mc.mean(dim=0)
            predictions = torch.argmax(avg_logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    return all_labels, all_predictions

def save_predictions_to_csv(labels, predictions, filename='predictions.csv'):
    df = pd.DataFrame({'True Label': labels, 'Predicted Label': predictions})
    df.to_csv(filename, index=False)
    print(f"Predictions saved to {filename}")

def send_telegram_message(title, message):
    load_dotenv('.env')
    token = os.getenv('TELEGRAM_BOT_TOKEN')

    try:
        response = requests.post(f'https://api.telegram.org/bot{token}/sendMessage', data={
            'chat_id': os.getenv('TELEGRAM_CHAT_ID'),
            'text': f'{title}\n{message}',
            #'parse_mode': 'Markdown'
        })
    except requests.exceptions.RequestException as e:
        print(f"Error sending message: {e}")
        return None


import torch
import pyro
import pyro.distributions as dist
from pyro.nn.module import PyroModule, PyroParam
from pyro.infer.autoguide import AutoGuide
from pyro.infer.autoguide.initialization import InitMessenger, init_to_feasible
from pyro.distributions import constraints
from contextlib import ExitStack

from pyro.distributions.util       import sum_rightmost
from pyro.ops.tensor_utils         import periodic_repeat
from pyro.distributions.transforms import biject_to
from pyro.infer.autoguide.utils    import (
    deep_setattr,
    deep_getattr,
    helpful_support_errors,
)

import pyro.poutine as poutine

In [7]:
num_classes = 2

pyro.clear_param_store()

In [8]:
bayesian_model = BayesShipsCNNSmartpool(num_classes,
        device,
        activation="relu",
        prior_dist="gaussian",
        mu = 0.0,
        b= 1.0,
        smartpool_switch = False,
        pool_threshold=10.0,
        pool_detect_only=False,
        dropout_switch=False,
        dropout_p=0.5
        #prior_params={'mu': 0.0, 'b': b_iter})
        )

[INFO] Using prior: gaussian (mu=0.0, b=1.0)


In [29]:
import pyro
from pyro.infer.autoguide import AutoGuideList  #, AutoLowRankMultivariateNormal\
from pyro.infer.autoguide import AutoLowRankMultivariateNormal
from pyro import poutine

# assume `model` is your BayesShipsCNNSmartpool instance
guide = AutoGuideList(bayesian_model)

# 1) conv1.weight
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["conv1.weight"]),
        rank=20,
        init_scale=0.05,
        #prefix="AutoGuideList.conv1.weight"
    )
)

# 2) conv1.bias
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["conv1.bias"]),
        rank=5,
        init_scale=0.05,
        #prefix="AutoGuideList.conv1.bias"
    )
)

# 3) conv2.weight
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["conv2.weight"]),
        rank=20,
        init_scale=0.05,
        #prefix="AutoGuideList.conv2.weight"
    )
)

# 4) conv2.bias
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["conv2.bias"]),
        rank=5,
        init_scale=0.05,
        #prefix="AutoGuideList.conv2.bias"
    )
)

# 5) fc1.weight
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["fc1.weight"]),
        rank=20,
        init_scale=0.05,
        #prefix="AutoGuideList.fc1.weight"
    )
)

# 6) fc1.bias
guide.add(
    AutoLowRankMultivariateNormal(
        poutine.block(bayesian_model, expose=["fc1.bias"]),
        rank=5,
        init_scale=0.05,
        #prefix="AutoGuideList.fc1.bias"
    )
)


In [15]:
from pyro.optim import ClippedAdam

optimizer = ClippedAdam({"lr": 1e-3, "weight_decay": 0.0})

In [16]:
svi = pyro.infer.SVI(model=bayesian_model,
                    guide=guide,
                    optim=optimizer,
                    loss=pyro.infer.Trace_ELBO(num_particles=1,
                                                )) #TODO

pyro.clear_param_store()

In [17]:
bayesian_model.to(device)
guide.to(device)

AutoGuideList(
  (0-5): 6 x AutoLowRankMultivariateNormal()
)

In [18]:
train_loader, test_loader = load_data(batch_size=16)

In [19]:
losses, accuracies, accuracy_epochs, loc_stats, scale_stats, best_model_path, best_guide_path, best_param_store_path, experiment_timestamp = train_svi_with_stats(
bayesian_model,
guide,
svi,
train_loader,
device,
num_epochs=1,
save_epochs=None,
save_dir="results_guidemultivariate",)

Epoch 1/1: 100%|██████████| 200/200 [00:39<00:00,  5.09it/s]


Epoch 1 - ELBO Loss: 112028.3727


Acc check epoch 1: 100%|██████████| 200/200 [00:03<00:00, 58.99it/s]


Epoch 1 - Train Acc: 88.16%
  ↳ Saved: model_relu_gaussian_epoch_best_20250722_205306.pth, guide_relu_gaussian_epoch_best_20250722_205306.pth, param_store_relu_gaussian_epoch_best_20250722_205306.pkl


In [20]:
# print pyro param store name and values
for name, param in pyro.get_param_store().items():
    print(f"{name}: {param.shape}")

AutoGuideList.0.loc: torch.Size([864])
AutoGuideList.0.scale: torch.Size([864])
AutoGuideList.0.cov_factor: torch.Size([864, 20])
AutoGuideList.1.loc: torch.Size([32])
AutoGuideList.1.scale: torch.Size([32])
AutoGuideList.1.cov_factor: torch.Size([32, 5])
AutoGuideList.2.loc: torch.Size([18432])
AutoGuideList.2.scale: torch.Size([18432])
AutoGuideList.2.cov_factor: torch.Size([18432, 20])
AutoGuideList.3.loc: torch.Size([64])
AutoGuideList.3.scale: torch.Size([64])
AutoGuideList.3.cov_factor: torch.Size([64, 5])
AutoGuideList.4.loc: torch.Size([32768])
AutoGuideList.4.scale: torch.Size([32768])
AutoGuideList.4.cov_factor: torch.Size([32768, 20])
AutoGuideList.5.loc: torch.Size([2])
AutoGuideList.5.scale: torch.Size([2])
AutoGuideList.5.cov_factor: torch.Size([2, 5])


In [28]:
# make a list of layer for my model, do it automatically from my model layer name

pyro_guide_mapping = {
    "0": "conv1.weight",
    "1": "conv1.bias",
    "2": "conv2.weight",
    "3": "conv2.bias",
    "4": "fc1.weight",
    "5": "fc1.bias"
}

pyro_guide_title_mapping = {"AutoGuideList": "AutoLowRankMultivariateNormal"}

In [11]:
#activation_fn,prior,best_accuracy,prior_mu,prior_b,param_type,location_index,location_layer,location_module,bit_index,initial_accuracy,accuracy_after_seu,accuracy_change,softmax_difference,mean_abs_difference
#relu,gaussian,0.8640625,0.0,1.0,scales,0,conv1,weight,1,0.92125,0.75,-0.17125,1.0,1.3632405825697595e+38

In [12]:
def return_accuracy(all_labels, all_predictions):
    cm = confusion_matrix(all_labels, all_predictions)
    return np.trace(cm) / np.sum(cm)

def compute_softmax_difference(before_probs, after_probs):
    before_probs = np.array(before_probs)
    after_probs = np.array(after_probs)
    diff = np.abs(before_probs - after_probs)
    return np.max(diff, axis=1).mean()

def compute_difference(original_val, modified_val):
    return abs(original_val - modified_val)

def predict_data_probs(num_samples=10):
    all_labels = []
    all_predictions = []
    all_logits = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            logits_mc = torch.zeros(num_samples, images.size(0), bayesian_model.fc1.out_features).to(device)

            for i in range(num_samples):
                guide_trace = pyro.poutine.trace(guide).get_trace(images)
                replayed_model = pyro.poutine.replay(bayesian_model, trace=guide_trace)
                logits = replayed_model(images)
                logits_mc[i] = logits

            avg_logits = logits_mc.mean(dim=0)
            predictions = torch.argmax(avg_logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_logits.extend(avg_logits.cpu().numpy())
            all_probs.extend(F.softmax(avg_logits, dim=1).cpu().numpy())

    return all_labels, all_predictions, all_logits, all_probs

In [21]:
param = pyro.get_param_store().get_param('AutoGuideList.0.loc')
new_param = param.clone()

In [22]:
from bitflip import bitflip_float32

In [23]:
new_param = new_param.view(-1) #flatten new param
original_val = new_param[0].cpu().item()
seu_val = bitflip_float32(original_val, 1)

In [24]:
abs_diff = compute_difference(original_val, seu_val)
new_param[0] = seu_val
# return new_param to original shape
new_param = new_param.view(param.shape)
pyro.get_param_store().__setitem__('AutoGuideList.0.loc', new_param)


In [25]:
print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")

Original value: -0.22181038558483124, SEU value: -7.547816301445238e+37, Abs difference: 7.547816301445238e+37


In [26]:
num_samples = 10

after_labels, after_predictions, after_logits, after_probs = predict_data_probs(num_samples)
accuracy_after = return_accuracy(after_labels, after_predictions)
#softmax_diff = self.compute_softmax_difference(self.initial_probs, after_probs)

Evaluating: 100%|██████████| 50/50 [00:32<00:00,  1.53it/s]


In [27]:
accuracy_after

np.float64(0.92125)